# [IAPR][iapr]: Final project - Chocolate Recognition


**Moodle group ID:** *xx*  
**Kaggle challenge:** *xx* (either `Classic` or `Deep learning`)  
**Kaggle team name (exact):** "*xx*"  

**Author 1 (SCIPER):** *Student Name 1 (xxxxx)*  
**Author 2 (SCIPER):** *Student Name 2 (xxxxx)*  
**Author 3 (SCIPER):** *Student Name 3 (xxxxx)*  

**Due date:** 21.05.2025 (11:59 pm)


## Key Submission Guidelines:
- **Before submitting your notebook, <span style="color:red;">rerun</span> it from scratch!** Go to: `Kernel` > `Restart & Run All`
- **Only groups of three will be accepted**, except in exceptional circumstances.


[iapr]: https://github.com/LTS5/iapr2025

---

> Your comments  
> ...

# TORCH WORK

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
x = np.linspace(-10, 10, 400)
y = x ** 2
plt.figure(figsize=(6, 6))
plt.plot(x, y, label='y = x²')
plt.title('Graph of y = x²')
plt.xlabel('x')
plt.ylabel('y')
plt.grid(True)
plt.legend()
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.show()

In [1]:
from src.utils import YoloGridDataset, print_total_paramters, train_model, use_model_on_images_with_nms, use_model_on_image_with_nms_by_name, create_controller_csv, evaluate_f1_score,create_table_from_pt_nms
import pandas as pd
from src.models import YOLOv8Lite, TinyYOLO, CompactYOLOv2
from pathlib import Path
import os
import torch
from torch.utils.data import DataLoader

In [2]:

IMG_PARAM = {}
IMG_PARAM["IMG_SIZE"] = 640
IMG_PARAM["ANCHOR"] = 3
IMG_PARAM["NUM_CLASSES"] = 13

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = YOLOv8Lite().to(device)

if(isinstance(model,TinyYOLO) or isinstance(model,CompactYOLOv2)):
    IMG_PARAM["GRID_SIZE"] = 20
elif(isinstance(model,YOLOv8Lite)):
    IMG_PARAM["GRID_SIZE"] = 40


current_dir = os.getcwd()
train_root_dir = Path(f'{current_dir}/data/train')

# Use in your dataset
train_dataset = YoloGridDataset(
    root_dir=train_root_dir,
    GRID_SIZE=IMG_PARAM["GRID_SIZE"],
    image_size=(IMG_PARAM["IMG_SIZE"], IMG_PARAM["IMG_SIZE"])
)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
EPOCHS = 200
name_of_saved_pt = "YoloV8LiteAugmentedData"


test_root_dir = Path(f'{current_dir}/data/test')

test_dataset = YoloGridDataset(
    root_dir=test_root_dir,
    GRID_SIZE=IMG_PARAM["GRID_SIZE"],
    image_size=(IMG_PARAM["IMG_SIZE"], IMG_PARAM["IMG_SIZE"])
)

test_loader = DataLoader(test_dataset, batch_size=2, shuffle=True)

In [3]:

print_total_paramters(model)

Total Parameters: 10,227,126


In [4]:
model_save_path = f"{current_dir}/model_weights"
train_model(model=model, total_epochs=EPOCHS, optimizer=optimizer, device=device, 
            per_epoch_save=40, train_loader=train_loader, test_loader=test_loader, plotting_callback=None, 
            name_of_saved_pt=name_of_saved_pt, pt_save_path=model_save_path)

Epoch 2/200:   0%|          | 1/200 [03:35<11:54:34, 215.45s/it]

New best loss: 70.3605
Best model saved to: /home/ubuntu/Coding/EE-451/project/model_weights/YoloV8LiteAugmentedData_best.pt
Epoch 1/200, Total Loss: 70.3605


Epoch 3/200:   1%|          | 2/200 [07:09<11:48:14, 214.62s/it]

New best loss: 45.3826
Best model saved to: /home/ubuntu/Coding/EE-451/project/model_weights/YoloV8LiteAugmentedData_best.pt
Epoch 2/200, Total Loss: 45.3826


KeyboardInterrupt: 

In [ ]:
model_weights_path = f"{current_dir}/model_weights"

# Load the model only once
model = YOLOv8Lite()
model.load_state_dict(torch.load(f"{model_weights_path}/YoloV8LiteAugmentedData200.pt", map_location=torch.device("cpu")))

# Run inference
data_root_dir = Path(f'{current_dir}/data/')
use_model_on_images_with_nms(root_dir=data_root_dir, number_of_images=10, model=model, IMG_PARAM=IMG_PARAM, conf_threshold=0.13, min_dist=40.0)



In [ ]:
# Load the model only once
model = YOLOv8Lite()
model.load_state_dict(torch.load("models/YOLOv8Lite_epoch10.pt", map_location=torch.device("cpu")))

test_root_dir = Path(f'{current_dir}/test')
images = ["L1000995.jpg", "L1010042.jpg"]
use_model_on_image_with_nms_by_name(root_dir=test_root_dir, image_names=images, model=model, IMG_PARAM=IMG_PARAM, conf_threshold=0.1)


In [ ]:
create_controller_csv()

In [ ]:
evaluate_f1_score(pd.read_csv("DoNotSubmitThis_Controller/controller.csv"))

In [ ]:
# list_of_threshold = np.array(range(25))*0.01
# list_of_model_names = ["YOLOv8Lite_epoch100.pt","YOLOv8Lite_epoch200.pt","YOLOv8Lite_epoch300.pt"]

# list_of_threshold = np.array(range(0,6))*0.02
list_of_threshold = np.array(range(1,14))*0.01
list_of_model_names = ["YoloV8LiteAugmentedData200.pt"]
test_root_dir = Path(f'{current_dir}/data/test')
#epoch 200 conf 13 dist 40 -> 85

for model_name in list_of_model_names:
    for threshold in list_of_threshold:
        df = create_table_from_pt_nms(data_root_dir=test_root_dir, 
                                      model_path=f"{current_dir}/model_weights/{model_name}",
                                      model_class=YOLOv8Lite,
                                      IMG_PARAM=IMG_PARAM,
                                      conf_threshold=threshold,
                                      min_dist=40)
        print(f"Score for model {model_name}")
        evaluate_f1_score(df)

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# Your classes in order (matching YOLO IDs)
CLASS_NAMES = ['Amandina', 'Arabia', 'Comtesse', 'Creme_brulee', 'Jelly_Black',
               'Jelly_Milk', 'Jelly_White', 'Noblesse', 'Noir_authentique',
               'Passion_au_lait', 'Stracciatella', 'Tentation_noir', 'Triangolo']

# Paths
image_dir = "train_annotated_augmented/images"
label_dir = "train_annotated_augmented/labels"

# Get some image filenames
image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.png'))]
image_files = image_files[:25]  # Show only first 5 for example

for image_file in image_files:
    image_path = os.path.join(image_dir, image_file)
    label_path = os.path.join(label_dir, image_file.replace('.jpg', '.txt').replace('.png', '.txt'))

    img = cv2.imread(image_path)
    h, w = img.shape[:2]

    if not os.path.exists(label_path):
        print(f"No label found for {image_file}")
        continue

    with open(label_path, 'r') as f:
        for line in f.readlines():
            cls, x, y, box_w, box_h = map(float, line.strip().split())

            # Convert from relative to absolute coords
            x1 = int((x - box_w / 2) * w)
            y1 = int((y - box_h / 2) * h)
            x2 = int((x + box_w / 2) * w)
            y2 = int((y + box_h / 2) * h)

            class_name = CLASS_NAMES[int(cls)]

            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img, class_name, (x1, max(y1 - 10, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 0), 2)

    # Convert BGR to RGB for matplotlib
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.figure(figsize=(8, 8))
    plt.imshow(img_rgb)
    plt.title(f"Labels: {image_file}")
    plt.axis('off')
    plt.show()